In [1]:
import scanpy as sc
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts


In [2]:
atlas_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/atlas_data.h5ad")
query_data = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/LCA_h5ads/Meyer_2021_raw.h5ad")

### set labels to use for integration:

check which cells have only level 1 annotations:

In [3]:
lev1_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 1, :].copy()
print("Number of cells with only level 1 annotation:", lev1_obs.shape[0])

Number of cells with only level 1 annotation: 0


In [4]:
# count number of cells per ann_level_1 group that have no further annotations
lev1_annotations = lev1_obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).rename(columns={"ann_level_1":"n_cells_w_only_l1_ann"})
# count total cells in each ann_level_1 group
lev1_annotations['total'] = query_data.obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).loc[lev1_annotations.index,"ann_level_1"]

In [5]:
lev1_annotations

,n_cells_w_only_l1_ann,total
ann_level_1,,
Endothelial,0,21282
Epithelial,0,22569
Glial,0,36
Immune,0,70364
Stroma,0,14377


Not applicable here. (We will let all "Proliferating cells" keep their labels, and will set all other cells as "unlabeled" (since they only make up a small part of their group))

In [6]:
lev1_labeled = query_data.obs.index[query_data.obs.ann_level_1 == "Proliferating cells"]

In [7]:
lev1_unlabeled = query_data.obs.index[[max_res == 1 and lev1label != "Proliferating cells" for max_res, lev1label in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_1)]]

In [8]:
len(lev1_labeled)

0

In [9]:
len(lev1_unlabeled)

0

In [10]:
# check if all labels/cells are accounted for
lev1_annotations.n_cells_w_only_l1_ann.sum() == len(lev1_labeled) + len(lev1_unlabeled)

True

Now check which cells have only up to level 2 annotations:

In [11]:
lev2_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 2, :].copy()
print("Number of cells with only level 1/2 annotation:", lev2_obs.shape[0])

Number of cells with only level 1/2 annotation: 3027


In [12]:
# count number of cells per ann_level_2 group that have no further annotations
lev2_annotations = lev2_obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).rename(columns={"ann_level_2":"n_cells_w_only_l2_ann"})
# count total cells in each ann_level_2 group
lev2_annotations['total'] = query_data.obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).loc[lev2_annotations.index,"ann_level_2"]

In [13]:
lev2_annotations

,n_cells_w_only_l2_ann,total
ann_level_2,,
Airway epithelium,0,11689
Alveolar epithelium,0,8632
Blood vessels,0,18364
Chondrocytes,42,42
Fibroblast lineage,0,12279
Lymphatic EC,2918,2918
Lymphoid,0,31157
Megakaryocytic and erythroid,0,339
Mesothelium,67,67


We will set all cells with level 2 annotations "Chondrocytes", "Fibroblast lineage", "Schwann cells", "Lymphatic EC", "Mesothelium", "Smooth muscle" to their level 2 annotations.

In [14]:
lev2_labels_to_keep = ["Chondrocytes", "Fibroblast lineage", "Schwann cells", "Lymphatic EC", "Mesothelium", "Smooth muscle"]
lev2_labeled = query_data.obs.index[[lev2_lab in lev2_labels_to_keep for lev2_lab in query_data.obs.ann_level_2]]

In [15]:
len(lev2_labeled) == lev2_annotations.loc[lev2_labels_to_keep,"total"].sum()

True

In [16]:
lev2_unlabeled = query_data.obs.index[[max_res == 2 and lev2_lab not in lev2_labels_to_keep for max_res, lev2_lab in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_2)]]

In [17]:
len(lev2_unlabeled) == lev2_annotations.loc[~lev2_annotations.index.isin(lev2_labels_to_keep),"n_cells_w_only_l2_ann"].sum()

True

now generate scANVI labels:

In [18]:
query_data.obs['scanvi_label'] = query_data.obs.ann_level_3.tolist()
query_data.obs.loc[lev2_labeled,'scanvi_label'] = query_data.obs.loc[lev2_labeled, "ann_level_2"]
query_data.obs.loc[query_data.obs.ann_level_3=="Innate Lymphoid Cell NK","scanvi_label"] = "Innate lymphoid cell NK"
query_data.obs.loc[lev2_unlabeled,'scanvi_label'] = "unlabeled"

In [19]:
# Check which label in query data is new and not present in reference atlas
[x for x in query_data.obs['scanvi_label'].unique() if x not in atlas_data.obs["scanvi_label"].unique()]

['Erythrocytes', 'Schwann cells', 'Chondrocytes']

### Normalize counts

In [ ]:
#Save raw counts
query_data.layers["counts"] = query_data.X.copy()
# normalize 
query_data_norm = scib_excerpts.SCRAN_normalize(query_data, log_transform=False)

In [ ]:
# store
query_data_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Meyer2021/Meyer_2021_norm.h5ad")